In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
import warnings
import joblib

warnings.filterwarnings('ignore')
load_dotenv()

# ============================================================================
# Project paths
# ============================================================================
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR = RESULTS_DIR / 'model'
TABLE_DIR = RESULTS_DIR / 'table'

print("Project paths ready")

print("=" * 80)
print("Agent #1: CF Quality Interpreter (v5 — Stage 3 flags for review, no auto-overwrite)")
print("=" * 80)

# ============================================================================
# 1. Feature Name Mapping
# ============================================================================
BASE_TERM_MAPPING = {
    'FN1_1': 'Current Assets', 'FN1_2': 'Non-Current Assets', 'FN1_3': 'Quick Assets',
    'FN1_4': 'Inventory', 'FN1_5': 'Tangible Assets', 'FN1_6': 'Work in Process',
    'FN1_7': 'Cash', 'FN1_8': 'Cash Equivalents', 'FN1_9': 'Marketable Securities',
    'FN1_10': 'Cash and Cash Equivalents', 'FN1_11': 'Accounts Receivable',
    'FN1_11_2': 'Loss on Disposal of Receivables', 'FN1_11_3': 'Intangible Assets',
    'FN1_11_4': 'Investment Assets', 'FN1_14': 'Current Liabilities',
    'FN1_15': 'Short-Term Borrowings', 'FN1_16': 'Borrowings', 'FN1_17': 'Accounts Payable',
    'FN1_18': 'Non-Current Liabilities', 'FN1_19': 'Total Liabilities',
    'FN1_20': 'Paid-in Capital', 'FN1_21': 'Capital Surplus', 'FN1_21_1': 'Paid-in Capital (Detail)',
    'FN1_22': 'Retained Earnings', 'FN1_22_1': 'Capital Adjustments',
    'FN1_22_2': 'Accumulated Other Comprehensive Income', 'FN1_23': 'Reserves',
    'FN1_24': 'Total Equity', 'FN3_10_1': 'Liquidation Value', 'FN3_11': 'Net Working Capital',
    'FN3_11_1': 'Net Borrowings',
    'FN2_2': 'Cost of Goods Sold', 'FN2_2_1': 'Gross Profit', 'FN2_3': 'SG&A Expenses',
    'FN2_3_1': 'Pre-Tax Income', 'FN2_3_2': 'Prior-Year Pre-Tax Income', 'FN2_3_3': 'Corporate Tax',
    'FN2_3_4': 'Income from Continuing Operations', 'FN2_3_5': 'Discontinued Operations Gain/Loss',
    'FN2_4': 'Financial Expenses', 'FN2_5': 'Operating Income', 'FN2_5_1': 'Prior-Year Operating Income',
    'FN2_7': 'Non-Operating Income', 'FN2_8': 'Non-Operating Expenses', 'FN2_9': 'Pre-Tax Net Income',
    'FN2_10': 'Net Income', 'FN3_1': 'Cash Flow', 'FN3_2': 'Operating Cash Flow',
    'FN3_2_1': 'Investing Cash Flow', 'FN3_2_2': 'Financing Cash Flow', 'FN3_4_1': 'Interest Expense',
    'FN3_4_2': 'Bond Interest', 'FN3_7': 'EBIT', 'FN3_8': 'EBITDA',
    'FN3_3': 'Debt Service Coverage Ratio', 'FN3_6': 'Reserve Ratio', 'FN3_10': 'Liquidation Value Ratio',
    'asset_growth_rate': 'Total Asset Growth Rate', 'revenue_growth_rate': 'Revenue Growth Rate',
    'operating_income_growth': 'Operating Income Growth Rate', 'net_income_growth': 'Net Income Growth Rate',
    'equity_growth_rate': 'Equity Growth Rate',
}

def get_readable_name(feature_name):
    if feature_name in BASE_TERM_MAPPING:
        return BASE_TERM_MAPPING[feature_name]
    if feature_name.endswith('_to_assets'):
        base = feature_name[:-len('_to_assets')]
        base_term = BASE_TERM_MAPPING.get(base, base)
        return f"{base_term} (% of Total Assets)"
    if feature_name.endswith('_to_revenue'):
        base = feature_name[:-len('_to_revenue')]
        base_term = BASE_TERM_MAPPING.get(base, base)
        return f"{base_term} (% of Revenue)"
    return feature_name

# ============================================================================
# 2. Industry Context
# ============================================================================
INDUSTRY_CONTEXT = {
    'G46': {
        'name': 'wholesale trade',
        'risk_factors': 'elevated accounts-receivable exposure and inventory concentration risk',
        'mitigating_factors': 'established trade-credit relationships and inventory that can typically be liquidated or renegotiated over a 1-2 quarter horizon'
    },
    'G47': {
        'name': 'retail trade',
        'risk_factors': 'thinner margins and high inventory turnover pressure',
        'mitigating_factors': 'faster cash conversion cycles than wholesale, giving retailers more frequent opportunities to adjust pricing and inventory levels'
    },
    'L68': {
        'name': 'real estate',
        'risk_factors': 'asset-heavy balance sheets and liquidity constrained by illiquid property holdings',
        'mitigating_factors': 'typically substantial collateral value that supports refinancing or partial asset disposal as a recourse path'
    },
    'F42': {
        'name': 'construction',
        'risk_factors': 'project-based cash flow volatility and contract-completion risk',
        'mitigating_factors': 'milestone-based billing and retention receivables that can often be accelerated or factored to improve near-term liquidity'
    },
}
DEFAULT_INDUSTRY = {
    'name': "the firm's sector", 'risk_factors': 'sector-specific financial risk characteristics',
    'mitigating_factors': 'standard recourse mechanisms available to firms in comparable sectors'
}

def get_industry_context(sic_code):
    return INDUSTRY_CONTEXT.get(sic_code, DEFAULT_INDUSTRY)

# ============================================================================
# 3. STAGE 1 — Deterministic Pre-Check (unchanged)
# ============================================================================
PASS_PROXIMITY_MAX = 0.15
PASS_SPARSITY_MIN = 0.5

def stage1_deterministic_check(row):
    proximity = row['Proximity']
    sparsity = row['Sparsity']
    proximity_ok = proximity < PASS_PROXIMITY_MAX
    sparsity_ok = sparsity > PASS_SPARSITY_MIN
    eligible = proximity_ok and sparsity_ok
    reason = (f"Proximity={proximity:.4f} ({'OK' if proximity_ok else 'exceeds'} {PASS_PROXIMITY_MAX} threshold), "
              f"Sparsity={sparsity:.4f} ({'OK' if sparsity_ok else 'below'} {PASS_SPARSITY_MIN} threshold)")
    return eligible, reason

# ============================================================================
# 4. STAGE 3 — Independent Audit (v2, stricter prompt). IMPORTANT: this stage
#    now ONLY FLAGS for human review — it no longer auto-overwrites Stage 2's
#    text. Rationale: an earlier version that auto-applied Stage 3's
#    "corrections" was found to introduce false positives (flagging
#    stylistically-different but semantically-correct text as erroneous).
#    Rather than keep tuning the audit prompt indefinitely, flagged cases are
#    now surfaced for a human reviewer, consistent with the paper's
#    Human-in-the-Loop design philosophy (Section 5.3) — AI drafts and flags,
#    a human makes the final call on anything flagged.
# ============================================================================
def stage3_full_recheck(client, model_name, business_meaning, feasibility_assessment,
                         industry, proximity, sparsity, eligible_for_pass):
    recheck_prompt = f"""
    You are a strict auditor reviewing a financial analyst's written report
    for TWO specific, narrowly-defined failure modes. You must NOT flag
    stylistic imperfections, awkward phrasing, or synonyms — ONLY flag an
    actual reversal of meaning.

    [Explanation to review — business_meaning]
    {business_meaning}

    [Explanation to review — feasibility_assessment]
    {feasibility_assessment}

    [Ground truth metrics]
    Proximity = {proximity:.4f} (LOWER is BETTER. Below {PASS_PROXIMITY_MAX} = GOOD.)
    Sparsity = {sparsity:.4f} (HIGHER is BETTER. Above {PASS_SPARSITY_MIN} = GOOD.)
    Correct Pass/Warning label = {'PASS' if eligible_for_pass else 'WARNING'}

    [Sector context]
    Risk factors: {industry['risk_factors']}
    Mitigating factors: {industry['mitigating_factors']}

    Task 1 — One-sided framing: does the text mention risk factors while
    COMPLETELY omitting mitigating factors, or the reverse? (Only flag if one
    side is entirely absent — do not flag if both are mentioned even briefly.)

    Task 2 — Numeric misinterpretation: flag this ONLY if the text does ONE
    of these four SPECIFIC things — otherwise leave it as false, even if the
    wording is awkward, redundant, or could be phrased more elegantly:
      (a) Calls a Proximity value BELOW {PASS_PROXIMITY_MAX} "risky," "concerning,"
          "problematic," or similarly negative — a low Proximity is GOOD, so this
          is a genuine reversal.
      (b) Calls a Proximity value ABOVE {PASS_PROXIMITY_MAX} "minor," "achievable,"
          or similarly positive — a high Proximity is BAD, so this is a genuine reversal.
      (c) Calls a Sparsity value BELOW {PASS_SPARSITY_MIN} "consistent with an
          approved profile," "feasible," "manageable," or similarly POSITIVE —
          a low Sparsity is BAD, so calling it positive is a genuine reversal.
          (Note: correctly calling a low Sparsity value "not good," "concerning,"
          or negative is CORRECT and must NOT be flagged.)
      (d) Calls a Sparsity value ABOVE {PASS_SPARSITY_MIN} "concerning" or
          "problematic" — a high Sparsity is GOOD, so this is a genuine reversal.
    Two sentences that use different words but agree on WHICH DIRECTION is good
    or bad are NOT an error. Only flag an actual reversal of which direction is favorable.

    Respond in JSON:
    {{
        "is_one_sided": true/false,
        "has_numeric_misinterpretation": true/false,
        "misinterpretation_pattern": "If flagged, which of (a)/(b)/(c)/(d) applies and the exact quoted phrase. Otherwise null.",
        "suggested_business_meaning": "A suggested correction ONLY if a genuine reversal was found; otherwise null.",
        "suggested_feasibility_assessment": "A suggested correction ONLY if a genuine reversal was found; otherwise null."
    }}
    """
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a precise auditor who flags ONLY genuine reversals of meaning — never stylistic differences, redundancy, or awkward phrasing. When in doubt, do NOT flag."},
                {"role": "user", "content": recheck_prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        return {
            "error": str(e), "is_one_sided": False, "has_numeric_misinterpretation": False,
            "misinterpretation_pattern": None,
            "suggested_business_meaning": None, "suggested_feasibility_assessment": None
        }

# ============================================================================
# 5. Data Loading
# ============================================================================
df_filtered = pd.read_csv(DATA_DIR / 'cf_results_filtered_4industry.csv')
df_details = pd.read_csv(DATA_DIR / 'cf_evaluation_details_4industry.csv')
selected_features = joblib.load(MODEL_DIR / 'selected_features_final_full.pkl')

print(f"Analysis targets (optimal CFs): {len(df_filtered)} records")
print(f"Reference pool (all candidates): {len(df_details)} records")

unmapped = [f for f in selected_features if get_readable_name(f) == f and f not in BASE_TERM_MAPPING]
if unmapped:
    print(f"\n[Warning] {len(unmapped)} features have no readable mapping: {unmapped}")

# ============================================================================
# 6. Agent Class Definition
# ============================================================================
class CFInterpreter:
    def __init__(self):
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model_stage2 = os.getenv('LLM_MODEL', 'gpt-4o-mini')
        self.model_stage3 = os.getenv('LLM_MODEL_STAGE3_AUDIT', 'gpt-4o')
        print(f"Stage 2 (narrative generation) model: {self.model_stage2}")
        print(f"Stage 3 (audit / flag-only) model:     {self.model_stage3}")

    def generate_interpretation(self, best_row, all_candidates):
        company_id = best_row['ID']
        sic_code = best_row.get('SIC_CD_3', None)
        industry = get_industry_context(sic_code)

        other_cfs = all_candidates[all_candidates['ID'] == company_id]
        avg_score = other_cfs['Quality_Score'].mean()
        best_score = best_row['Quality_Score']
        proximity = best_row['Proximity']
        sparsity = best_row['Sparsity']

        # ---------------- STAGE 1: deterministic pre-check ----------------
        eligible_for_pass, stage1_reason = stage1_deterministic_check(best_row)

        changes = {}
        for feat in selected_features:
            delta = best_row[f'Change_{feat}']
            if abs(delta) > 0.001:
                readable_name = get_readable_name(feat)
                changes[readable_name] = {
                    'original': round(float(best_row[f'Original_{feat}']), 4),
                    'target': round(float(best_row[f'CF_{feat}']), 4),
                    'delta': round(float(delta), 4)
                }
        sorted_changes = dict(sorted(changes.items(), key=lambda item: abs(item[1]['delta']), reverse=True))
        n_changed = len(sorted_changes)
        n_total_features = len(selected_features)

        # ---------------- STAGE 2: narrative generation, with directional
        # guardrail ----------------------------------------------------------
        system_prompt = f"""
        You are a corporate credit insurance underwriting analyst specializing
        in the {industry['name']} sector. A quantitative screen has already
        determined whether this scenario qualifies for "Pass" or "Warning."
        Your task is to explain the numbers and the sector context — you do
        NOT decide the Pass/Warning label, only justify it, and you must
        reference both the sector's risk factors ({industry['risk_factors']})
        and its mitigating factors ({industry['mitigating_factors']}).

        DIRECTIONAL GUARDRAIL — apply these interpretation rules exactly:
        - Proximity measures how MUCH the firm's financials had to change.
          LOWER Proximity is GOOD (less change needed). Never describe a low
          Proximity value as "close to risk" or "aligned with risk factors" —
          low Proximity means the scenario is EASY to achieve, not risky.
        - Sparsity measures the SHARE of variables left UNCHANGED. HIGHER
          Sparsity is GOOD (fewer variables needed adjustment). Never
          describe a low Sparsity value as "consistent with an approved
          profile" — low Sparsity means MANY variables changed, which is
          the opposite of a minimal, easily-achievable scenario.
        """

        user_prompt = f"""
        [Company ID: {company_id} | Sector: {industry['name']}] — AI Financial Improvement Proposal

        # Quantitative Pre-Check Result (already determined, do not override)
        Label: {'PASS' if eligible_for_pass else 'WARNING'}
        Basis: {stage1_reason}
        Reminder: Proximity {proximity:.4f} is {'BELOW' if proximity < PASS_PROXIMITY_MAX else 'ABOVE'} the
            {PASS_PROXIMITY_MAX} threshold, which is {'GOOD' if proximity < PASS_PROXIMITY_MAX else 'NOT GOOD'}.
            Sparsity {sparsity:.4f} is {'ABOVE' if sparsity > PASS_SPARSITY_MIN else 'BELOW'} the
            {PASS_SPARSITY_MIN} threshold, which is {'GOOD' if sparsity > PASS_SPARSITY_MIN else 'NOT GOOD'}.

        # 1. Selection Rationale
        Selected from {len(other_cfs)} candidates with quality score {best_score:.2f}
        (pool average {avg_score:.2f}). Realism={best_row['Realism']:.2f},
        Robustness={best_row['Robustness']:.2f}.
        Only {n_changed} of {n_total_features} financial variables required any
        meaningful change; the rest are already consistent with an approved profile.

        # 2. Proposed Changes, Largest to Smallest
        {json.dumps(sorted_changes, ensure_ascii=False, indent=2)}

        # Task
        Return JSON:
        {{
            "selection_reason": "One sentence, citing quality indicators.",
            "business_meaning": "2-3 sentences on what the changes mean in practice, referencing BOTH the sector's risk factors AND its mitigating factors from the persona instructions above. Follow the DIRECTIONAL GUARDRAIL exactly.",
            "feasibility_assessment": "Must start with the word '{'Pass' if eligible_for_pass else 'Warning'}' exactly, followed by a brief justification citing the Proximity/Sparsity basis given above, following the DIRECTIONAL GUARDRAIL exactly.",
            "expected_outcome": "Expected positive outcomes if implemented."
        }}
        """

        try:
            response = self.client.chat.completions.create(
                model=self.model_stage2,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"},
                temperature=0.3
            )
            stage2_result = json.loads(response.choices[0].message.content)
        except Exception as e:
            stage2_result = {"error": str(e)}

        # ---------------- STAGE 3: audit, flag-only (no auto-overwrite) ----
        stage3_result = {
            "is_one_sided": None, "has_numeric_misinterpretation": None,
            "misinterpretation_pattern": None,
            "suggested_business_meaning": None, "suggested_feasibility_assessment": None,
        }
        if 'business_meaning' in stage2_result and 'feasibility_assessment' in stage2_result:
            stage3_result = stage3_full_recheck(
                self.client, self.model_stage3,
                stage2_result['business_meaning'], stage2_result['feasibility_assessment'],
                industry, proximity, sparsity, eligible_for_pass
            )
            # NOTE: stage2_result['business_meaning'] / ['feasibility_assessment']
            # are NEVER overwritten here. Stage 3's findings are stored
            # separately for human review, per the Human-in-the-Loop design.

        stage2_result['stage1_eligible_for_pass'] = eligible_for_pass
        stage2_result['stage1_reason'] = stage1_reason
        stage2_result['stage3_was_one_sided'] = stage3_result.get('is_one_sided')
        stage2_result['stage3_flagged_for_review'] = stage3_result.get('has_numeric_misinterpretation')
        stage2_result['stage3_flag_pattern'] = stage3_result.get('misinterpretation_pattern')
        stage2_result['stage3_suggested_business_meaning'] = stage3_result.get('suggested_business_meaning')
        stage2_result['stage3_suggested_feasibility'] = stage3_result.get('suggested_feasibility_assessment')
        stage2_result['stage2_model'] = self.model_stage2
        stage2_result['stage3_model'] = self.model_stage3

        return stage2_result

# ============================================================================
# 7. Execution and Export
# ============================================================================
agent = CFInterpreter()
results = []

print(f"\nGenerating interpretations for {len(df_filtered)} firms (Stage 3 = flag-only audit)...")

from tqdm import tqdm
for idx, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    interpretation = agent.generate_interpretation(row, df_details)

    result_entry = row.to_dict()
    result_entry.update(interpretation)
    results.append(result_entry)

final_df = pd.DataFrame(results)

output_path = DATA_DIR / 'agent1_interpretation_results_4industry.csv'
final_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print("\n" + "=" * 80)
print("Interpretation generation complete")
print("=" * 80)
print(f"Output file: {output_path.relative_to(PROJECT_ROOT)}")

# ============================================================================
# 8. Distribution and audit checks
# ============================================================================
print("\n[Stage 1 — deterministic Pass/Warning distribution]")
print(final_df['stage1_eligible_for_pass'].value_counts())
print(f"Pass rate: {final_df['stage1_eligible_for_pass'].mean()*100:.1f}%")

print("\n[Stage 1 — by industry]")
print(pd.crosstab(final_df['SIC_CD_3'], final_df['stage1_eligible_for_pass']))

print("\n[Stage 3 — one-sided framing flagged]")
print(final_df['stage3_was_one_sided'].value_counts(dropna=False))

print("\n[Stage 3 — flagged for human review (numeric)]")
print(final_df['stage3_flagged_for_review'].value_counts(dropna=False))
print(f"Flag rate: {final_df['stage3_flagged_for_review'].mean()*100:.1f}%")

if final_df['stage3_flagged_for_review'].sum() > 0:
    print("\n[Sample flagged cases — for human reviewer]")
    flagged_samples = final_df[final_df['stage3_flagged_for_review'] == True][
        ['ID', 'SIC_CD_3', 'feasibility_assessment', 'stage3_flag_pattern', 'stage3_suggested_feasibility']
    ].head(5)
    print(flagged_samples.to_string(index=False))

# Save a dedicated review queue for anything Stage 3 flagged
review_queue = final_df[
    (final_df['stage3_was_one_sided'] == True) | (final_df['stage3_flagged_for_review'] == True)
]
if not review_queue.empty:
    review_path = TABLE_DIR / 'agent1_human_review_queue.csv'
    review_queue.to_csv(review_path, index=False, encoding='utf-8-sig')
    print(f"\n{len(review_queue)} cases saved to human review queue: {review_path.relative_to(PROJECT_ROOT)}")

print("\n[Sample outputs by industry]")
for sic in ['G46', 'G47', 'L68', 'F42']:
    sector_rows = final_df[final_df['SIC_CD_3'] == sic]
    if not sector_rows.empty:
        sample = sector_rows.iloc[0]
        print(f"\n--- {sic} (ID: {sample['ID']}) ---")
        print(f"Stage1 eligible_for_pass: {sample['stage1_eligible_for_pass']}")
        print(f"1. Selection reason: {sample.get('selection_reason')}")
        print(f"2. Business meaning: {sample.get('business_meaning')}")
        print(f"3. Feasibility assessment: {sample.get('feasibility_assessment')}")
        print(f"   (Stage3 one-sided: {sample.get('stage3_was_one_sided')}, flagged for review: {sample.get('stage3_flagged_for_review')})")

Project paths ready
Agent #1: CF Quality Interpreter (v5 — Stage 3 flags for review, no auto-overwrite)
Analysis targets (optimal CFs): 542 records
Reference pool (all candidates): 2443 records
Stage 2 (narrative generation) model: gpt-4o-mini
Stage 3 (audit / flag-only) model:     gpt-4o

Generating interpretations for 542 firms (Stage 3 = flag-only audit)...


100%|████████████████████████████████████████████████████████████████████████████████| 542/542 [54:02<00:00,  5.98s/it]


Interpretation generation complete
Output file: data\agent1_interpretation_results_4industry.csv

[Stage 1 — deterministic Pass/Warning distribution]
stage1_eligible_for_pass
False    454
True      88
Name: count, dtype: int64
Pass rate: 16.2%

[Stage 1 — by industry]
stage1_eligible_for_pass  False  True 
SIC_CD_3                              
F42                          76     20
G46                         154     34
G47                         114     24
L68                         110     10

[Stage 3 — one-sided framing flagged]
stage3_was_one_sided
False    542
Name: count, dtype: int64

[Stage 3 — flagged for human review (numeric)]
stage3_flagged_for_review
False    497
True      45
Name: count, dtype: int64
Flag rate: 8.3%

[Sample flagged cases — for human reviewer]
  ID SIC_CD_3                                                                                                                                                                                                     